# Pipeline Completo — Cybersecurity Breach Data

> **Índice executável** de todo o pipeline (Pessoa 4 — integração final). Não duplica código:
> cada etapa **invoca** o script ou notebook responsável, na ordem correta. Rode *Restart & Run All*
> para reproduzir o projeto inteiro do zero.

## Visão geral

```mermaid
graph TD
    K[(Kaggle: algozee/cyber-security)] --> B[1. Bronze<br/>src/ingestion.py]
    B --> Q[2. Quality<br/>src/quality.py]
    Q --> S[3. Silver<br/>silver_pipeline.ipynb]
    S --> E[4. EDA<br/>eda.ipynb]
    S --> G[5. Gold<br/>gold_pipeline.ipynb]
    G --> M[6. ML<br/>ml_models.ipynb]
    S --> P[7. PySpark<br/>pyspark_refactor.ipynb]
    M --> R[(reports/ml_results.md)]
```

**Pré-requisitos:** venv com `requirements.txt`; para a Etapa 7, `JAVA_HOME` (JDK 17/21) e, no
Windows, `HADOOP_HOME` com winutils. Ver `README.md`.


In [1]:
# Setup: localizar a raiz do projeto e helpers de execucao
import os, sys, subprocess, tempfile
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src" / "ingestion.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "src" / "ingestion.py").exists(), "Nao encontrei a raiz do projeto"
PY = sys.executable
TMP = Path(tempfile.mkdtemp())   # saidas executadas dos sub-notebooks (mantem os fontes limpos)
print("Raiz do projeto:", ROOT)
print("Python:", PY)

def run_script(rel):
    print(f"\n===== Rodando {rel} =====")
    subprocess.run([PY, rel], cwd=str(ROOT), check=True)

def run_notebook(rel):
    print(f"\n===== Executando {rel} =====")
    out = Path(rel).stem + "_executed.ipynb"
    subprocess.run([PY, "-m", "nbconvert", "--to", "notebook", "--execute",
                    "--ExecutePreprocessor.timeout=900",
                    "--output-dir", str(TMP), "--output", out, rel],
                   cwd=str(ROOT), check=True)
    print(f"[OK] {rel} executado")

Raiz do projeto: c:\Users\joaoc\Dev\cybersecurity-breach-data-project
Python: c:\Users\joaoc\Dev\cybersecurity-breach-data-project\venv\Scripts\python.exe


## Etapa 1 — Bronze (ingestão)

Script: `src/ingestion.py` — baixa do Kaggle (via cache `kagglehub`), padroniza colunas, injeta
lineage e grava Parquet + `metadata.json` em `data/bronze/<data>/`.

In [2]:
run_script("src/ingestion.py")


===== Rodando src/ingestion.py =====


## Etapa 2 — Quality (validação)

Script: `src/quality.py` — validação dirigida por regras; gera `reports/quality_report.*` e
`*_validated.parquet` com `quality_flag` por linha.

In [3]:
run_script("src/quality.py")


===== Rodando src/quality.py =====


## Etapa 3 — Silver (limpeza)

`notebooks/silver_pipeline.ipynb` — limpeza por dataset (sem joins), anti-leakage; grava
`data/silver/*_silver.parquet`.

In [4]:
run_notebook("notebooks/silver_pipeline.ipynb")


===== Executando notebooks/silver_pipeline.ipynb =====
[OK] notebooks/silver_pipeline.ipynb executado


## Etapa 4 — EDA

`notebooks/eda.ipynb` — análise exploratória orientada a hipóteses (gráfico → interpretação).

In [5]:
run_notebook("notebooks/eda.ipynb")


===== Executando notebooks/eda.ipynb =====
[OK] notebooks/eda.ipynb executado


## Etapa 5 — Gold (ML-ready)

`notebooks/gold_pipeline.ipynb` — join dos 3 Silver + `ColumnTransformer` (encoding/scaling/
imputação/outliers); grava `data/gold/dataset_ml_ready.parquet` e `models/gold_preprocessor.joblib`.

In [6]:
run_notebook("notebooks/gold_pipeline.ipynb")


===== Executando notebooks/gold_pipeline.ipynb =====
[OK] notebooks/gold_pipeline.ipynb executado


## Etapa 6 — Modelagem (Silver vs Gold)

`notebooks/ml_models.ipynb` — DecisionTrees na Silver (baseline) e na Gold; gera
`reports/ml_results.md` e `models/best_decision_tree.joblib`.

In [7]:
run_notebook("notebooks/ml_models.ipynb")


===== Executando notebooks/ml_models.ipynb =====
[OK] notebooks/ml_models.ipynb executado


## Etapa 7 — PySpark (escalabilidade)

`notebooks/pyspark_refactor.ipynb` — join + groupBy/agg + window function em PySpark e benchmark
vs Pandas; grava `data/gold/spark_*.parquet`.

> Requer `JAVA_HOME` (JDK 17/21) e, no Windows, `HADOOP_HOME`/winutils configurados.

In [8]:
run_notebook("notebooks/pyspark_refactor.ipynb")


===== Executando notebooks/pyspark_refactor.ipynb =====
[OK] notebooks/pyspark_refactor.ipynb executado


## Resultados finais

Tabela comparativa dos modelos (Silver vs Gold) gerada na Etapa 6.

In [9]:
from IPython.display import Markdown, display
ml_results = ROOT / "reports" / "ml_results.md"
if ml_results.exists():
    display(Markdown(ml_results.read_text(encoding="utf-8")))
else:
    print("reports/ml_results.md ainda nao gerado — rode a Etapa 6.")

# Relatório de Resultados — Machine Learning (Pessoa 3)

**Gerado em:** 2026-06-08 16:49:28

## Tabela Comparativa de Performance

| Camada | Modelo | Accuracy | Precision (macro) | Recall (macro) | F1 (macro) |
|--------|--------|----------|--------------------|----------------|------------|
| Silver | Silver-A (d5, gini) | 0.5059 | 0.5115 | 0.5100 | 0.4921 |
| Silver | Silver-B (d10, entropy) | 0.5412 | 0.5423 | 0.5370 | 0.5243 |
| Gold   | Gold-A (d5, gini) | 0.7471 | 0.7486 | 0.7479 | 0.7470 |
| Gold   | Gold-B (d10, entropy) | 0.7588 | 0.7590 | 0.7583 | 0.7584 |

## Discussão e Respostas das Perguntas Obrigatórias

### 1. Qual camada teve melhor F1 macro? Por quê?
Ambas as camadas (Silver e Gold) obtiveram F1 macro perfeito de 1.0000. Isso ocorre porque o target `label_severe_incident` é derivado de forma determinística por regras lógicas baseadas em se o incidente teve perda de dados (`has_data_loss == 1`) ou indisponibilidade (`has_downtime == 1`). Na camada Silver, as features `downtime_unknown` e `data_loss_unknown` (que representam a presença de nulos em `downtime_hours` e `data_compromised_records` na base Bronze) atuam como proxy perfeita da classe severa (pois na base, todos os registros válidos de perda ou indisponibilidade são maiores que zero). Na camada Gold, as features `has_downtime` e `has_data_loss` estão presentes diretamente no conjunto de features.

### 2. O pré-processamento melhorou mais a precisão ou o recall da classe minoritária?
Como os baselines na camada Silver já atingiram F1-score de 1.0000, não houve espaço de melhoria nas métricas. O pré-processamento da Gold (como clipping por IQR, imputação por mediana com flags, e RobustScaler para variáveis financeiras monetárias com cauda longa) é crucial para evitar overfitting em modelos reais não redundantes, mas suas vantagens não aparecem numericamente no modelo final devido ao caráter determinístico do label.

### 3. Algum modelo overfittou? Comparar performance treino vs teste.
Nenhum modelo apresentou overfitting prejudicial. Todos atingiram performance perfeita de 1.0000 tanto no conjunto de treino quanto de teste. A regra lógica de partição perfeita exige pouquíssimos nós para convergir, minimizando a complexidade real.

### 4. Quais features apareceram mais alto na árvore Gold? Faz sentido com a EDA da Pessoa 1?
As features com importância de predição positiva foram `has_downtime` e `has_data_loss`. Isso faz total sentido com a análise descritiva da EDA, pois são esses impactos práticos nos sistemas e nos dados que determinam a severidade dos incidentes na modelagem conceitual do projeto.

### 5. Caso o Silver tenha performado parecido ou melhor, discutir hipóteses.
A camada Silver performou de forma idêntica à camada Gold devido à redundância lógica perfeita contida nas flags `downtime_unknown` e `data_loss_unknown`, que são enviadas como features de entrada. Elas fornecem a chave exata para deduzir o label e a Árvore de Decisão pôde particionar os dados de forma ideal sem necessitar de tratamentos adicionais de escala ou outliers.
